# Detecting Bubbles Inside the Two Main Rectangles

This notebook continues the OMR pipeline. The goal here is not final grading yet. The goal is to reach a clean intermediate result: detect the two largest sheet rectangles, warp both of them, and then detect answer-bubble candidates inside each warped section.

The paper we are following uses a classical computer vision pipeline before any machine-learning idea:

1. preprocess the scanned sheet,
2. detect the two main reference rectangles,
3. correct perspective using those rectangles,
4. find option circles/marks inside each section,
5. later decide whether each answer is selected, blank, or multiple.

Here we implement the same spirit in a compact way for our project. Later, the bubble candidates from this notebook can feed either a rule-based detector or an AI model.

In [ ]:
from pathlib import Path
import os

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

from preprocessing import (
    resize_image,
    image_to_grayscale,
    add_gaussian_blur,
    apply_canny_edge_detection,
)

from counter_detection import (
    prepare_rectangle_detection_image,
    find_external_contours,
    select_largest_rectangular_contours,
    build_warped_rectangle_sections,
    draw_all_contours,
    draw_corner_points,
)

from bubble_detection import (
    detect_bubble_candidates_hough,
    filter_bubbles_by_color_presence,
    score_bubble_marks,
    draw_bubbles,
    draw_mark_scores,
    threshold_dark_marks,
)

In [ ]:
def load_rgb(path):
    return np.array(Image.open(path).convert("RGB"))


def show_image(image, title="", ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 7))

    if image.ndim == 2:
        ax.imshow(image, cmap="gray")
    else:
        ax.imshow(image)

    ax.set_title(title)
    ax.axis("off")
    return ax


def show_grid(items, columns=2, figsize=(14, 9)):
    rows = int(np.ceil(len(items) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=figsize)
    axes = np.array(axes).reshape(-1)

    for ax, (title, image) in zip(axes, items):
        show_image(image, title, ax=ax)

    for ax in axes[len(items):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


def draw_section_labels(image, sections):
    image_copy = image.copy()

    for section in sections:
        points = section["corner_points"].astype(int)
        x, y = points[0]
        cv2.putText(
            image_copy,
            f"{section['name']} | area={section['area']:.0f}",
            (x, max(25, y - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 0, 0),
            2,
            cv2.LINE_AA,
        )

    return image_copy

## 1. Load the Scanned Sheet

We use one of the Tamaulipas-style scanned sheets. These are scanner images, so the problem is different from phone-camera perspective correction, but the rectangle detection stage is still useful.

In [ ]:
source_path = Path("samples/File0005.jpg")
original_image = load_rgb(source_path)

show_image(original_image, f"Original scanned sheet: {source_path.name}")
plt.show()

print("Original shape:", original_image.shape)

## 2. Preprocess the Image

The default preprocessing size is useful for early tests, but for bubble work we keep more detail. The image is resized to a taller working canvas, then converted to grayscale, blurred, and passed through Canny. We also save the Canny result because it is useful as a visual checkpoint.

In [ ]:
working_size = (900, 1200)  # PIL/OpenCV convention here: width, height

resized_image = resize_image(original_image, size=working_size)
grayscale_image = image_to_grayscale(resized_image)
blurred_image = add_gaussian_blur(grayscale_image)
canny_image = apply_canny_edge_detection(blurred_image)

canny_output_path = Path("samples/canny_File0005.png")
cv2.imwrite(str(canny_output_path), canny_image)

show_grid([
    ("Resized image", resized_image),
    ("Grayscale", grayscale_image),
    ("Gaussian blur", blurred_image),
    ("Canny edges", canny_image),
], columns=2, figsize=(12, 10))

print(f"Saved Canny image to: {canny_output_path}")

## 3. Detect the Two Largest Rectangles

The paper detects the ID section and answer section as the two main rectangles. Here we create a binary image for rectangle detection, find external contours, and keep the two largest rectangle-like contours.

In [ ]:
rectangle_binary = prepare_rectangle_detection_image(grayscale_image)
rectangle_contours = find_external_contours(rectangle_binary)
main_rectangles = select_largest_rectangular_contours(
    rectangle_contours,
    count=2,
    min_area=5000,
)

sections = build_warped_rectangle_sections(
    resized_image,
    main_rectangles,
    count=2,
)

print("Detected main rectangles:", len(sections))
for section in sections:
    print(section["name"], "area=", round(section["area"), "warp shape=", section["warped_image"].shape)

rectangles_overlay = draw_all_contours(
    resized_image,
    main_rectangles,
    color=(255, 0, 0),
    thickness=4,
)
rectangles_overlay = draw_section_labels(rectangles_overlay, sections)

show_grid([
    ("Binary image for rectangle detection", rectangle_binary),
    ("Two largest rectangle-like contours", rectangles_overlay),
], columns=2, figsize=(13, 7))

## 4. Warp Both Rectangles

Previously we only inspected the biggest rectangle. Here we also warp the second biggest one. In this sample, `rectangle_1` is the main answer section and `rectangle_2` is the student ID section.

In [ ]:
for section in sections:
    output_path = Path(f"samples/{section['name']}_warped_File0005.png")
    cv2.imwrite(str(output_path), cv2.cvtColor(section["warped_image"], cv2.COLOR_RGB2BGR))
    print(f"Saved {section['name']} warp to: {output_path}")

show_grid(
    [(f"Warped {section['name']}", section["warped_image"]) for section in sections],
    columns=1,
    figsize=(10, 12),
)

## 5. Detect Bubble Candidates in Each Warped Rectangle

For now we use a classical Hough-circle baseline. This is not the final detector, but it gives us a good set of candidate bubbles. Later we can replace or compare this with the paper-style rule detector and with an AI detector.

In [ ]:
section_results = []

for section in sections:
    warped = section["warped_image"]
    min_side = min(warped.shape[:2])

    bubbles = detect_bubble_candidates_hough(
        warped,
        min_distance=max(8, int(min_side * 0.02)),
        min_radius=max(3, int(min_side * 0.006)),
        max_radius=max(10, int(min_side * 0.02)),
        param2=12,
    )

    filtered_bubbles = filter_bubbles_by_color_presence(
        warped,
        bubbles,
        min_color_ratio=0.08,
    )

    scored_bubbles = score_bubble_marks(
        warped,
        filtered_bubbles,
        inner_radius_ratio=0.75,
        dark_threshold=130,
    )

    section_results.append({
        **section,
        "bubbles": bubbles,
        "filtered_bubbles": filtered_bubbles,
        "scored_bubbles": scored_bubbles,
    })

    print(
        section["name"],
        "raw candidates=", len(bubbles),
        "filtered candidates=", len(filtered_bubbles),
        "strong marks=", sum(item["dark_ratio"] >= 0.40 for item in scored_bubbles),
    )

## 6. Visualize Candidate Bubbles

Green circles are detected bubble candidates. The left image shows raw Hough candidates. The right image shows the candidates after a simple color-presence filter, which removes many false detections on row numbers.

In [ ]:
candidate_views = []

for result in section_results:
    raw_candidate_view = draw_bubbles(
        result["warped_image"],
        result["bubbles"],
        color=(0, 255, 0),
        thickness=2,
    )

    filtered_candidate_view = draw_bubbles(
        result["warped_image"],
        result["filtered_bubbles"],
        color=(0, 255, 0),
        thickness=2,
    )

    candidate_views.append((f"{result['name']} - raw candidates", raw_candidate_view))
    candidate_views.append((f"{result['name']} - filtered candidates", filtered_candidate_view))

show_grid(candidate_views, columns=2, figsize=(14, 12))

## 7. Visualize Strongly Marked Candidates

This is the first simple algorithmic mark detector: inside each candidate bubble, count how many pixels are dark. Red circles are candidates whose dark-pixel ratio is high enough to look selected. This is not final grading yet, but it is already useful for debugging.

In [ ]:
mark_views = []

for result in section_results:
    dark_marks = threshold_dark_marks(result["warped_image"], dark_threshold=130)
    score_view = draw_mark_scores(
        result["warped_image"],
        result["scored_bubbles"],
        mark_threshold=0.40,
        candidate_color=(0, 255, 0),
        marked_color=(255, 0, 0),
        thickness=2,
    )

    mark_views.append((f"{result['name']} - dark pixels", dark_marks))
    mark_views.append((f"{result['name']} - strong marks in red", score_view))

show_grid(mark_views, columns=2, figsize=(14, 12))

## Current Checkpoint

At this point, we have exactly the stage we wanted for the next part of the project:

- the largest rectangle is detected and warped,
- the second largest rectangle is detected and warped,
- bubble candidates are detected inside both rectangles,
- dark/filled candidates can be highlighted with a simple algorithm.

The next clean step is grouping bubbles into rows and columns. After that, we can compare two strategies: a classical rule-based detector and an AI/ML detector.